# Data Cleaning and Feature Generation

This notebook shows how the raw ChemFluor dataset is cleaned and converted into machine learning features.

In [2]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT))
print("Project root:", PROJECT_ROOT)

Project root: c:\Users\CL\OneDrive\Desktop\python\ChemFluor_Project_synced


## Load and Clean the Dataset

The training dataset can be stored as `data/chemfluor_data.csv` or, for the older layout, as root-level `chemfluor_data.csv`.

Required columns:

- `SMILES`
- `solvent`
- `Emission/nm`
- `PLQY`

In [4]:
from src.data import load_raw_data, clean_data

raw = load_raw_data()
print("Raw shape:", raw.shape)
raw.head()

Loading dataset: C:\Users\CL\OneDrive\Desktop\python\ChemFluor_Project_synced\data\chemfluor_data.csv
Raw shape: (4386, 113)


,Unnamed: 0,Absorption/nm,Emission/nm,PLQY,SMILES,solvent,Reference(doi),Et30,SP,SdP,...,Unnamed: 103,Unnamed: 104,Unnamed: 105,Unnamed: 106,Unnamed: 107,Unnamed: 108,Unnamed: 109,Unnamed: 110,Unnamed: 111,Unnamed: 112
0,1,355.0,427.0,0.390,O=C(OC)C1=CC2=NC(C3=CC=C(C=C3)OC)=CN2C=C1,DMSO,10.1021ja204016e_0306,45.1,0.83,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2,426.0,520.0,0.430,O=C(OC)C1=CC2=NC(C3=CC=C(C=C3)OC)=C(N2C=C1)N,DMSO,10.1021ja204016e_0306,45.1,0.83,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3,387.0,528.0,0.490,O=C(C1=CC2=NC(C3=CC=C(OC)C=C3)=C(NC4CCCCC4)N2C...,DMSO,10.1021ja204016e_0306,45.1,0.83,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,4,355.0,627.0,0.002,COC1=CC=C(C=C1)C2=C(NC3CCCCC3)N(C(C(OC)=O)=CC=...,DMSO,10.1021ja204016e_0306,45.1,0.83,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,5,365.0,560.0,0.009,COC1=CC=C(C=C1)C2=C(NC3CCCCC3)N(C=CC=N4)C4=N2,DMSO,10.1021ja204016e_0306,45.1,0.83,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [5]:
cleaned, stats = clean_data(raw)
print("Cleaned shape:", cleaned.shape)
stats

Raw rows: 4386
Dropped 1296 rows missing SMILES, solvent, wavelength, or PLQY.
Dropped 0 rows with non-numeric targets.
Exact duplicate rows found: 0
Duplicate canonical molecule-solvent rows found: 32
Merged duplicate molecule-solvent rows by averaging targets: 32
Cleaned rows: 3058
Cleaned shape: (3058, 5)


{'raw_rows': 4386,
 'rows_after_required_dropna': 3090,
 'cleaned_rows': 3058,
 'invalid_smiles': 0,
 'exact_duplicates': 0,
 'merged_duplicate_pairs': 32,
 'unique_molecules': 1818,
 'unique_solvents': 52}

In [ ]:
cleaned[["SMILES", "canonical_smiles", "solvent", "Emission/nm", "PLQY"]].head()

## Build the Feature Matrix

This step creates Morgan fingerprints, MACCS keys, RDKit descriptors, one-hot solvent features, and solvent descriptor features if `data/solvent_descriptors.csv` or root-level `solvent_descriptors.csv` is available.

In [ ]:
from src.features import build_feature_matrix

X = build_feature_matrix(cleaned)
print("Feature matrix shape:", X.shape)
X.head()

In [ ]:
print("Number of Morgan features:", sum(c.startswith("morgan_") for c in X.columns))
print("Number of MACCS features:", sum(c.startswith("maccs_") for c in X.columns))
print("Number of solvent one-hot features:", sum(c.startswith("solvent_") for c in X.columns))
print("Number of solvent descriptor features:", sum(c.startswith("solvdesc_") for c in X.columns))

## Notes

If the solvent descriptor file is blank or missing, the model will still run using one-hot solvent features only. If descriptor values are partially missing, they are reported and median-imputed.